# MSHA data validation and regional panel

This Gate 4 notebook validates the frozen raw snapshot and builds the mine-quarter and state-quarter checkpoints. It does not select, train, or evaluate a forecasting model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Set one private Drive root below. It must contain `data/raw/MinesProdQuarterly.txt` and `data/raw/Mines.txt`. Checkpoints and run manifests will be written under the same private root.

In [ ]:
from pathlib import Path

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ds_portfolio/project_02_coal_production_forecasting')
DATA_ROOT = DRIVE_PROJECT_ROOT / 'data'
RUNS_ROOT = DRIVE_PROJECT_ROOT / 'runs'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import subprocess
import sys

REPO_URL = 'https://github.com/ahmaddshbg-blip/regional-coal-production-forecasting.git'
REPO_DIR = Path('/content/regional-coal-production-forecasting')
if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR)], check=True)
revision = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True
).stdout.strip()
print('Code revision:', revision)

In [ ]:
import os

os.chdir(REPO_DIR)
os.environ['PROJECT_DATA_ROOT'] = str(DATA_ROOT)
os.environ['PROJECT_RUNS_ROOT'] = str(RUNS_ROOT)

required_raw_files = [
    DATA_ROOT / 'raw' / 'MinesProdQuarterly.txt',
    DATA_ROOT / 'raw' / 'Mines.txt',
]
missing = [str(path) for path in required_raw_files if not path.is_file()]
if missing:
    raise FileNotFoundError('Place the manually downloaded raw files at: ' + ', '.join(missing))
print('Raw inputs found.')

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=REPO_DIR, check=True
)

In [ ]:
from coal_forecasting import build_dataset

OVERWRITE_EXISTING_CHECKPOINTS = False
manifest = build_dataset(
    REPO_DIR / 'configs' / 'project.json',
    overwrite=OVERWRITE_EXISTING_CHECKPOINTS,
    root=REPO_DIR,
)
{
    'run_id': manifest['run_id'],
    'status': manifest['status'],
    'manifest_path': manifest['manifest_path'],
    'outputs': {
        name: {key: value for key, value in details.items() if key in {'path', 'rows', 'bytes', 'sha256'}}
        for name, details in manifest['outputs'].items()
    },
}

In [ ]:
import duckdb

state_panel = DATA_ROOT / manifest['outputs']['state_quarter']['path']
with duckdb.connect() as connection:
    panel_summary = connection.execute(
        '''
        SELECT
            MIN(period) AS first_period,
            MAX(period) AS last_period,
            COUNT(*) AS state_quarter_rows,
            COUNT(DISTINCT state_code) AS states,
            COUNT(*) FILTER (WHERE coal_production_short_tons IS NULL) AS null_targets
        FROM read_parquet(?)
        ''',
        [str(state_panel)],
    ).fetchdf()
panel_summary

In [ ]:
with duckdb.connect() as connection:
    national_series = connection.execute(
        '''
        SELECT quarter_start_date, SUM(coal_production_short_tons) AS production_short_tons
        FROM read_parquet(?)
        GROUP BY quarter_start_date
        ORDER BY quarter_start_date
        ''',
        [str(state_panel)],
    ).fetchdf()

ax = national_series.plot(
    x='quarter_start_date', y='production_short_tons', figsize=(12, 4), legend=False
)
ax.set(title='U.S. coal production represented in the MSHA snapshot', xlabel='', ylabel='Short tons')